# Best-of-N SD1.5 + ImageReward — GPU / Colab

Sinh **N ảnh độc lập**, chạy đủ **T bước**, chấm ảnh cuối bằng ImageReward và chọn điểm **cao nhất**. Không scout, không sửa latent, không resampling giữa chừng. Mặc định N=24,T=100 → **2400 sample NFE**,4800 CFG-branch sample evaluations.

Chọn CUDA GPU runtime. Đây là bản độc lập dùng Diffusers, không phải bản replay bitwise của backend TPU cũ. Muốn đối chiếu đúng stack TPU, dùng notebook `best_of_n_sd15_current_tpu.ipynb`.

Code được cung cấp chưa kèm output neural giả lập. Chạy các cell lần lượt; model downloads có thể cần quyền đọc Hugging Face. Không ghi token vào notebook.

## 1. Cài trong runtime GPU riêng

Không chạy cell pip này trong môi trường TPU đang dùng. Nếu pip yêu cầu restart kernel, restart rồi chạy tiếp từ cell imports. Torch/Torchvision do runtime CUDA cung cấp; không thay bằng CPU wheels.

In [ ]:
get_ipython().run_line_magic('pip', 'install -q "diffusers==0.35.1" "transformers==4.38.2" "accelerate==1.2.1" "safetensors==0.5.2" "image-reward==1.5" "Pillow>=10"')

## 2. Helpers và tham số

In [ ]:
import csv,hashlib,importlib.metadata,json,math,time,uuid
from pathlib import Path
import numpy as np
import torch
from PIL import Image,ImageDraw
from IPython.display import display

def versions():
    result={}
    for package in ['torch','torchvision','diffusers','transformers','image-reward','numpy','Pillow','torch_xla']:
        try:result[package]=importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:pass
    return result

class UNetMeter:
    """Count actual samples through UNet, separately from batched forward calls."""
    def __init__(self,cfg_branches=2):
        self.branches=cfg_branches;self.sample_nfe=0;self.cfg_sample_evaluations=0;self.forward_calls=0
    def hook(self,module,args,kwargs):
        x=args[0] if args else kwargs['sample']
        count=int(x.shape[0]);assert count%self.branches==0
        self.sample_nfe+=count//self.branches
        self.cfg_sample_evaluations+=count;self.forward_calls+=1
    def as_dict(self):
        return dict(sample_nfe=self.sample_nfe,cfg_branch_sample_evaluations=self.cfg_sample_evaluations,
                    actual_batched_unet_forward_calls=self.forward_calls)

def write_result(prompt,images,scores,metadata,candidate_keys,out_root):
    scores=np.asarray(scores,dtype=np.float64).reshape(-1)
    assert len(images)==len(scores)==len(candidate_keys)>0 and np.isfinite(scores).all()
    # Stable tie-break: candidate with smaller index wins equal ImageReward.
    order=sorted(range(len(scores)),key=lambda i:(-float(scores[i]),i));best=order[0]
    folder=Path(out_root)/('run_'+time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]);folder.mkdir(parents=True,exist_ok=False)
    rows=[]
    for i,image in enumerate(images):
        path=folder/f'candidate_{i:03d}.png';image.save(path)
        rows.append(dict(candidate_id=i,candidate_key=str(candidate_keys[i]),image_reward=float(scores[i]),
                         image=path.name,image_sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    with (folder/'scores.csv').open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    images[best].save(folder/'best.png')
    cols=min(6,len(images));side=160;lines=math.ceil(len(images)/cols)
    sheet=Image.new('RGB',(cols*side,lines*(side+27)),'white');draw=ImageDraw.Draw(sheet)
    for pos,i in enumerate(order):
        x=(pos%cols)*side;y=(pos//cols)*(side+27)
        sheet.paste(images[i].resize((side,side)),(x,y))
        draw.text((x+3,y+side+3),f'#{i} IR={float(scores[i]):.4f}',fill='black')
    sheet.save(folder/'ranked_grid.jpg',quality=92)
    summary=dict(method='best_of_n',prompt=prompt,best_candidate_id=best,best_image_reward=float(scores[best]),
                 mean_image_reward=float(scores.mean()),ranking=order,candidates=rows,**metadata)
    (folder/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False,default=str)+'\n')
    print(f'Best candidate #{best}, ImageReward={scores[best]:.6f}; output: {folder}')
    print(json.dumps(metadata['accounting'],indent=2));display(images[best]);display(sheet)
    return summary,folder

In [ ]:
PROMPT = 'A red canoe floating on a still alpine lake with yellow autumn trees and a distant snow-covered mountain.'
N = 24
T = 100
CFG = 7.5
DDIM_ETA = 1.0  # DDIM noise eta; this baseline has NO correction eta.
BATCH_SIZE = 4
BASE_SEED = 2026102101
HEIGHT = WIDTH = 512
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
MODEL_REVISION = '451f4fe16113bff5a5d2269ed5ad43b0592e9a14'
REWARD_DEVICE = 'cpu'  # Avoid sharing GPU VRAM with the generator.
OUT_ROOT = Path('bon_outputs_gpu')
assert N>0 and T>0 and BATCH_SIZE>0 and CFG>1
print(f'BoN budget: {N} x {T} = {N*T} sample NFE; CFG branches = {2*N*T}')

## 3. Load SD1.5 và reward model

In [ ]:
from diffusers import StableDiffusionPipeline,DDIMScheduler
import ImageReward as RM
assert torch.cuda.is_available(),'Select a CUDA GPU runtime before loading models.'
DEVICE='cuda'
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
start=time.perf_counter()
pipe=StableDiffusionPipeline.from_pretrained(
    MODEL_ID,revision=MODEL_REVISION,torch_dtype=DTYPE,use_safetensors=True,
    safety_checker=None,feature_extractor=None,requires_safety_checker=False,
).to(DEVICE)
pipe.scheduler=DDIMScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)
pipe.unet.requires_grad_(False);pipe.vae.requires_grad_(False);pipe.text_encoder.requires_grad_(False)
pipe.unet.eval();pipe.vae.eval();pipe.text_encoder.eval()
model_load_seconds=time.perf_counter()-start
# Official ImageReward scoring; fixed one-image calls to avoid varying score batch shape.
start=time.perf_counter()
reward_model=RM.load('ImageReward-v1.0',device=REWARD_DEVICE)
reward_model.eval();reward_load_seconds=time.perf_counter()-start
print(dict(device=DEVICE,dtype=str(DTYPE),versions=versions()))

## 4. Sinh N ảnh

Mỗi candidate có CPU generator riêng, seed=BASE_SEED+candidate_id. Cùng generator đi qua cả initial noise và DDIM stochastic steps; không dùng chung một future tape cho toàn population. Microbatch cuối có thể nhỏ hơn BATCH_SIZE và không được padding. Đổi backend/dtype/batch có thể đổi kết quả số học dù seed giống nhau.

In [ ]:
@torch.inference_mode()
def sample_best_of_n_candidates(prompt,n,steps,batch_size,base_seed):
    meter=UNetMeter(cfg_branches=2)
    handle=pipe.unet.register_forward_pre_hook(meter.hook,with_kwargs=True)
    images=[];seeds=[int(base_seed)+i for i in range(n)]
    if DEVICE=='cuda':torch.cuda.synchronize()
    started=time.perf_counter()
    try:
        for offset in range(0,n,batch_size):
            batch_seeds=seeds[offset:offset+batch_size]
            generators=[torch.Generator(device='cpu').manual_seed(seed) for seed in batch_seeds]
            batch=pipe(prompt=[prompt]*len(batch_seeds),height=HEIGHT,width=WIDTH,
                       num_inference_steps=steps,guidance_scale=CFG,eta=DDIM_ETA,
                       generator=generators,num_images_per_prompt=1,output_type='pil').images
            assert len(batch)==len(batch_seeds)
            images.extend(batch)
            print(f'Generated {len(images)}/{n}')
    finally:
        handle.remove()
    if DEVICE=='cuda':torch.cuda.synchronize()
    seconds=time.perf_counter()-started
    assert len(images)==n and meter.sample_nfe==n*steps
    assert meter.cfg_sample_evaluations==2*n*steps
    return images,seeds,meter.as_dict(),seconds

images,candidate_seeds,sampling_accounting,generation_seconds=sample_best_of_n_candidates(
    PROMPT,N,T,BATCH_SIZE,BASE_SEED)

## 5. Chấm IR, chọn argmax, lưu ảnh/CSV/metadata

In [ ]:
start=time.perf_counter()
with torch.inference_mode():
    # Higher ImageReward is better. Do not choose the smallest score.
    scores=[float(np.asarray(reward_model.score(PROMPT,image)).reshape(-1)[0]) for image in images]
reward_seconds=time.perf_counter()-start
accounting=dict(**sampling_accounting,nominal_sample_nfe=N*T,physical_padding_sample_nfe=0,
                auxiliary_unet_nfe=0,scout_nfe=0,terminal_reward_scores=N,
                terminal_reward_calls=N,intermediate_reward_scores=0,
                note='Each sample NFE includes a CFG UNet evaluation with two branches; VAE/IR costs logged separately.')
summary,output_dir=write_result(PROMPT,images,scores,dict(
    backend='cuda_diffusers_standalone',model_id=MODEL_ID,model_revision=MODEL_REVISION,
    n=N,steps=T,cfg=CFG,ddim_eta=DDIM_ETA,dtype=str(DTYPE),batch_size=BATCH_SIZE,
    seeds=candidate_seeds,versions=versions(),scheduler=dict(pipe.scheduler.config),
    model_load_seconds=model_load_seconds,reward_load_seconds=reward_load_seconds,
    generation_seconds=generation_seconds,reward_seconds=reward_seconds,
    accounting=accounting,reward_api='ImageReward-v1.0 score per image, fixed batch1',
    comparability='Not bitwise comparable to TPU BF16/custom-transition experiments; pair baselines on this backend.'
),candidate_seeds,OUT_ROOT)

## Đọc kết quả

- `best.png`: ảnh được chọn.
- `candidate_*.png`: toàn bộ N ảnh, không chỉ winner.
- `ranked_grid.jpg`, `scores.csv`, `summary.json`: ranking, seeds, hashes và accounting.
- BoN@2400 với T=100 dùng N=24. BoN@2360 không thể có đúng số ảnh full100 nguyên; nếu so method2360 với BoN24=2400 phải ghi rõ chênh40NFE, không đổi T lặng lẽ.
- IR chỉ là objective selection; notebook không khẳng định semantic correctness hay diversity.

API nguồn: [Diffusers StableDiffusionPipeline v0.35.1](https://huggingface.co/docs/diffusers/v0.35.1/api/pipelines/stable_diffusion/text2img), [ImageReward official quick start](https://github.com/zai-org/ImageReward#quick-start).